# Clustering — K-Means, DBSCAN, Agglomerative, and GMM

**Author:** Shivani Bokka  
**Datasets:** Synthetic mall customers (segmentation demo), make_blobs / make_moons / make_circles (algorithm comparison on known shapes)  
**Goal:** Master every major clustering algorithm with evaluation metrics and business context

---

## What Is This Notebook About?

This notebook is a **complete walkthrough of clustering** — the most important family of unsupervised machine learning algorithms. Each section explains the core idea in plain English before diving into code.

Whether you're segmenting customers, grouping documents, or trying to find anomalies in sensor data, clustering is one of the most practical tools in any data scientist's toolkit.

---

## Algorithms Covered in This Notebook

| # | Algorithm | Key Idea |
|---|-----------|----------|
| 1 | K-Means | Assign points to the nearest centroid, then recompute centroids |
| 2 | Mini-Batch K-Means | K-Means on random mini-batches — faster for huge datasets |
| 3 | DBSCAN | Group points that are densely packed; label sparse points as noise |
| 4 | Agglomerative | Merge the closest pair of clusters repeatedly from the bottom up |
| 5 | Gaussian Mixture Models (GMM) | Soft probabilistic assignments using elliptical Gaussian distributions |

---

---

## Section 1 — Imports and Setup

Before anything else, we import every library we'll use throughout this notebook.

- **numpy / pandas** — core numerical and tabular data tools
- **matplotlib / seaborn** — all visualizations
- **sklearn** — clustering algorithms, metrics, and synthetic datasets
- **scipy** — hierarchical clustering dendrogram tools
- **warnings** — suppress noisy convergence warnings so output stays clean

We also set a **global random seed** (42) so that every result in this notebook is fully reproducible. If you run this notebook again, you will get the same numbers and plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

# Clustering algorithms
from sklearn.cluster import KMeans, MiniBatchKMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

# Evaluation metrics
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Synthetic datasets
from sklearn.datasets import make_blobs, make_moons, make_circles

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Scipy for dendrograms
from scipy.cluster.hierarchy import dendrogram, linkage

# Display settings
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# Global random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully!")
print(f"Random seed set to: {RANDOM_STATE}")

---

## Section 2 — What Is Clustering?

### The Core Idea

**Clustering** is a type of **unsupervised learning** — meaning we have data but no labels. There is no teacher telling the algorithm "this point belongs to group A" and "that point belongs to group B." Instead, the algorithm must discover the natural groupings entirely on its own, based solely on the structure of the data.

The goal is simple: **find groups of data points that are more similar to each other than they are to points in other groups.**

---

### How Clustering Differs from Classification

This is a common source of confusion:

| | Classification | Clustering |
|-|----------------|------------|
| **Labels** | Yes — provided during training | No — must be discovered |
| **Type** | Supervised learning | Unsupervised learning |
| **Goal** | Predict a known category for new data | Discover unknown natural groups |
| **Example** | Is this email spam or not? | What types of customers do I have? |

Think of it this way: in classification, someone has already sorted a pile of mail into "spam" and "not spam" and you learn from those examples. In clustering, you're handed an unsorted pile and asked to find the natural categories yourself.

---

### Real-World Applications

Clustering is everywhere in practice:

- **Customer Segmentation:** Group customers by purchase behavior, demographics, or engagement patterns so that marketing can target each group with tailored messaging.
- **Document Grouping:** Automatically organize thousands of articles, support tickets, or research papers into topic clusters — without reading them all manually.
- **Anomaly Detection:** Points that don't belong to any cluster (or form tiny isolated clusters) are likely anomalies — fraudulent transactions, broken sensors, or rare diseases.
- **Image Compression:** Replace many similar pixel colors with a single representative color per cluster (this is literally how K-Means is used in image quantization).
- **Genomics:** Group genes with similar expression patterns to find functionally related gene families.
- **Recommendation Systems:** Cluster users with similar tastes so recommendations can be made based on the whole cluster, not just individual history.

---

### A Note on Reproducibility

Many clustering algorithms involve **random initialization** — the starting positions of centroids, for example, affect the final result. This means two runs of the same algorithm on the same data can produce **different cluster assignments** if the random seed is different.

Throughout this notebook, we use `random_state=42` wherever possible to ensure every result you see is reproducible. We will also demonstrate what happens *without* a fixed seed in the K-Means instability section.

---

## Section 3 — Load and Explore Data

### Two Types of Data in This Notebook

**1. Synthetic Mall Customers Dataset**  
We generate 200 synthetic customers with three features: `age`, `annual_income`, and `spending_score`. This simulates the classic customer segmentation problem. We use `make_blobs` with 4 centers to create four realistic customer archetypes, then add a small amount of Gaussian noise to make it feel more realistic.

**2. Algorithm Comparison Datasets**  
We generate three standard benchmark datasets that are used throughout the ML community to test clustering algorithms:
- **make_blobs:** Well-separated, roughly spherical clusters. This is the "easy" case that most algorithms handle well.
- **make_moons:** Two crescent-shaped clusters that interlock. This tests whether an algorithm can find non-convex shapes.
- **make_circles:** Concentric rings. This is the extreme case of non-convex clusters — a simple circle inside a larger circle.

In [ ]:
# ── Mall Customers (4-cluster blobs) ─────────────────────────────────────────
np.random.seed(RANDOM_STATE)
X_mall_raw, _ = make_blobs(
    n_samples=200,
    n_features=3,
    centers=4,
    cluster_std=1.2,
    random_state=RANDOM_STATE
)

# Add meaningful noise and scale to realistic ranges
age          = np.clip(X_mall_raw[:, 0] * 8 + 38, 18, 75).astype(int)
annual_income = np.clip(X_mall_raw[:, 1] * 12 + 60, 15, 140).astype(int)
spending_score = np.clip(X_mall_raw[:, 2] * 15 + 50, 1, 100).astype(int)

df_mall = pd.DataFrame({
    'age': age,
    'annual_income': annual_income,
    'spending_score': spending_score
})

# Scale for clustering
scaler_mall = StandardScaler()
X_mall = scaler_mall.fit_transform(df_mall)

print("Mall Customers Dataset:")
print(f"  Shape: {df_mall.shape}")
print(f"  Features: {list(df_mall.columns)}")
print()
print(df_mall.describe().round(2))
print()

# ── Algorithm Comparison Datasets ────────────────────────────────────────────
X_blobs, y_blobs   = make_blobs(n_samples=300, centers=3, cluster_std=0.6,  random_state=RANDOM_STATE)
X_moons, y_moons   = make_moons(n_samples=300, noise=0.08,                  random_state=RANDOM_STATE)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5,  random_state=RANDOM_STATE)

# Scale all comparison datasets
X_blobs   = StandardScaler().fit_transform(X_blobs)
X_moons   = StandardScaler().fit_transform(X_moons)
X_circles = StandardScaler().fit_transform(X_circles)

print("Algorithm comparison datasets ready:")
print(f"  make_blobs:   {X_blobs.shape}")
print(f"  make_moons:   {X_moons.shape}")
print(f"  make_circles: {X_circles.shape}")

In [ ]:
# Plot all three synthetic benchmark datasets side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

datasets = [
    (X_blobs,   y_blobs,   'make_blobs\n(3 well-separated clusters)'),
    (X_moons,   y_moons,   'make_moons\n(2 crescent shapes)'),
    (X_circles, y_circles, 'make_circles\n(concentric rings)'),
]

for ax, (X, y, title) in zip(axes, datasets):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='Set1', s=25, alpha=0.8, edgecolors='none')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_aspect('equal')

plt.suptitle('Synthetic Datasets for Algorithm Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### How to Read This Chart: Synthetic Datasets for Algorithm Comparison

Each panel shows a 2D scatter plot of one of the three synthetic benchmark datasets. The colors represent the **true underlying cluster labels** (which we know because we generated the data ourselves). In a real clustering problem, we would not know these labels — this is just to help us visually verify that algorithms are finding the right groups.

**Panel 1 — make_blobs:** Three tight, well-separated clusters arranged in space. Every major clustering algorithm should handle this easily. This is the "easy mode" benchmark.

**Panel 2 — make_moons:** Two crescent-shaped clusters that are interleaved with each other. A straight line cannot separate them. Algorithms that assume spherical or convex clusters (like K-Means) will fail here. This dataset separates the good algorithms from the great ones.

**Panel 3 — make_circles:** One small cluster completely surrounded by a larger ring cluster. This is the hardest shape for most algorithms because the two groups are not linearly separable in any direction. Only density-based methods like DBSCAN can solve this correctly.

> **Key insight:** The shape of your data determines which algorithm you should use. Always visualize your data (or at least a 2D projection of it) before choosing a clustering method.

---

## Section 4 — K-Means: How It Works

### Lloyd's Algorithm (The Core Loop)

K-Means is the most widely used clustering algorithm. The core idea is elegantly simple and can be described in three steps:

1. **Initialize:** Place K centroids randomly in the feature space (we'll discuss better initialization shortly).
2. **Assign:** Assign every data point to the nearest centroid (using Euclidean distance).
3. **Update:** Recalculate each centroid as the mean of all points assigned to it.
4. **Repeat:** Go back to step 2 and repeat until the assignments stop changing (convergence).

Each iteration is guaranteed to reduce (or keep equal) the total within-cluster sum of squares (called **inertia**). The algorithm always converges, but it may converge to a **local minimum** — a solution that is good but not the best possible.

---

### K-Means++ Initialization

The standard random initialization often leads to poor local minima. **K-Means++** is a smarter initialization strategy:

1. Choose the first centroid uniformly at random from the data points.
2. For each subsequent centroid, choose a data point with **probability proportional to its squared distance from the nearest existing centroid**.
3. Repeat step 2 until K centroids are chosen.

**Why is this better?** K-Means++ spreads the initial centroids far apart across the data, dramatically reducing the chance of getting stuck in a bad local minimum. In practice, K-Means++ converges 2–10x faster than random initialization and almost always produces better final clusters.

Scikit-learn uses K-Means++ by default (`init='k-means++'`). **Always use it.**

---

### The `n_init` Parameter

Even with K-Means++, results can vary between runs. The `n_init` parameter controls how many times the entire algorithm is run with different initializations. The best result (lowest inertia) is kept.

- **`n_init=1`:** Run once. Fast, but fragile — you might get a bad result.
- **`n_init=10`:** Run ten times, keep the best. The scikit-learn default. Much more stable.
- **`n_init='auto'`:** In newer scikit-learn versions, this sets a sensible default based on context.

> **Rule of thumb:** Always use `n_init >= 10` and set `random_state` for reproducibility.

In [ ]:
# Demonstrate instability: run KMeans 5 times with n_init=1 and different seeds
print("=" * 55)
print("  K-Means instability: n_init=1, 5 different random seeds")
print("=" * 55)

inertias_unstable = []
for seed in [0, 7, 13, 42, 99]:
    km = KMeans(n_clusters=3, n_init=1, random_state=seed)
    km.fit(X_blobs)
    inertias_unstable.append(km.inertia_)
    print(f"  Seed={seed:3d}  |  Inertia={km.inertia_:.4f}")

print(f"\n  Range of inertias: {min(inertias_unstable):.4f} — {max(inertias_unstable):.4f}")
print(f"  Spread (max - min): {max(inertias_unstable) - min(inertias_unstable):.4f}")

print()
print("=" * 55)
print("  Stable result: n_init=10")
print("=" * 55)

inertias_stable = []
for seed in [0, 7, 13, 42, 99]:
    km = KMeans(n_clusters=3, n_init=10, random_state=seed)
    km.fit(X_blobs)
    inertias_stable.append(km.inertia_)
    print(f"  Seed={seed:3d}  |  Inertia={km.inertia_:.4f}")

print(f"\n  Range of inertias: {min(inertias_stable):.4f} — {max(inertias_stable):.4f}")
print(f"  Spread (max - min): {max(inertias_stable) - min(inertias_stable):.4f}")
print()
print("Conclusion: n_init=10 gives the same inertia regardless of seed.")
print("Always use n_init >= 10 and set random_state.")

---

## Section 5 — K-Means on Mall Customers Data

Now we apply K-Means to our synthetic mall customers dataset. We use 4 clusters because we generated the data with 4 centers. In Section 6, we'll show how to choose K when we don't already know the answer.

We use the first two features (age and annual income) for visualization since we can't easily plot in 3D. The algorithm still runs on all three features — we just display the result in 2D.

In [ ]:
# Train K-Means with 4 clusters on mall customers
km_mall = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
labels_mall = km_mall.fit_predict(X_mall)

# Add cluster labels back to the DataFrame for profiling later
df_mall['cluster'] = labels_mall

# Get centroids (in scaled space) then inverse-transform to original scale
centroids_scaled = km_mall.cluster_centers_
centroids_orig   = scaler_mall.inverse_transform(centroids_scaled)

print(f"K-Means (k=4) trained on mall customers.")
print(f"Inertia (within-cluster sum of squares): {km_mall.inertia_:.2f}")
print(f"Silhouette score: {silhouette_score(X_mall, labels_mall):.4f}")
print()
print("Cluster sizes:")
for c, n in zip(*np.unique(labels_mall, return_counts=True)):
    print(f"  Cluster {c}: {n} customers")

In [ ]:
# Scatter plot of clusters with centroids
palette = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Age vs Annual Income
for c in range(4):
    mask = labels_mall == c
    axes[0].scatter(df_mall.loc[mask, 'age'],
                    df_mall.loc[mask, 'annual_income'],
                    color=palette[c], label=f'Cluster {c}', alpha=0.7, s=50)
axes[0].scatter(centroids_orig[:, 0], centroids_orig[:, 1],
                marker='X', s=200, c='black', zorder=5, label='Centroids')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Annual Income ($k)')
axes[0].set_title('K-Means Clusters: Age vs Annual Income')
axes[0].legend()

# Right: Annual Income vs Spending Score
for c in range(4):
    mask = labels_mall == c
    axes[1].scatter(df_mall.loc[mask, 'annual_income'],
                    df_mall.loc[mask, 'spending_score'],
                    color=palette[c], label=f'Cluster {c}', alpha=0.7, s=50)
axes[1].scatter(centroids_orig[:, 1], centroids_orig[:, 2],
                marker='X', s=200, c='black', zorder=5, label='Centroids')
axes[1].set_xlabel('Annual Income ($k)')
axes[1].set_ylabel('Spending Score (1–100)')
axes[1].set_title('K-Means Clusters: Income vs Spending Score')
axes[1].legend()

plt.suptitle('K-Means Clustering — Mall Customers (k=4)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: K-Means Clusters

Each panel shows the same 200 customers, but projected onto two different pairs of features.

- **Each dot** is a customer. The dot's **color** indicates which cluster K-Means assigned them to.
- **Black X markers** are the cluster centroids — the average position of all customers in that cluster. Think of them as the "representative" customer for each group.
- **Left panel** shows Age vs Annual Income. Are older/younger customers segregated? Are income levels separating groups?
- **Right panel** shows Annual Income vs Spending Score. This is typically the most informative view for customer segmentation.

**What to look for:**
- Clusters with centroids that are far apart are well-separated — the algorithm is making meaningful distinctions.
- If you see a cluster that visually spans two different regions in the scatter plot, K-Means may be forcing unnatural groupings (this happens when the true data shape isn't spherical).
- The centroids are the key business output — they describe the "average" customer in each segment.

---

## Section 6 — Choosing K: Elbow Method and Silhouette Method

### The Fundamental Problem

K-Means requires you to specify K (the number of clusters) before you run it. But how do you know the right K? In a real business problem, there's no ground truth to look at.

Two of the most popular methods for choosing K are:

**1. The Elbow Method (Inertia)**  
Inertia is the sum of squared distances from each point to its assigned centroid. It always decreases as K increases (at K=N, each point is its own cluster and inertia = 0). The "elbow" is where the rate of decrease sharply slows down — adding more clusters beyond that point gives diminishing returns.

The challenge: the "elbow" is often subjective and not always visible.

**2. Silhouette Score**  
The silhouette score measures how similar a point is to its own cluster compared to other clusters:
- **Score close to +1:** The point is well inside its own cluster and far from all others. Perfect.
- **Score close to 0:** The point is near the boundary between two clusters. Ambiguous.
- **Score close to -1:** The point is closer to another cluster than its own. Likely misclustered.

The average silhouette score across all points gives a single number measuring overall cluster quality. **Higher is better.** Unlike inertia, there is a clear peak to look for.

> **Rule of thumb:** Use silhouette score as your primary metric; use the elbow plot as a sanity check.

In [ ]:
# Compute inertia and silhouette score for K from 2 to 12
K_range   = range(2, 13)
inertias  = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_mall)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_mall, labels))

# Print the table
print(f"{'K':>4}  {'Inertia':>12}  {'Silhouette':>12}")
print("-" * 32)
for k, iner, sil in zip(K_range, inertias, sil_scores):
    print(f"  {k:>2}  {iner:>12.2f}  {sil:>12.4f}")

best_k_sil = list(K_range)[sil_scores.index(max(sil_scores))]
print(f"\nBest K by silhouette score: K={best_k_sil} (score={max(sil_scores):.4f})")

In [ ]:
# Plot elbow and silhouette side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=11)
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4 (chosen)')
axes[0].legend()

# Silhouette plot
bar_colors = ['#e41a1c' if k == best_k_sil else 'steelblue' for k in K_range]
axes[1].bar(list(K_range), sil_scores, color=bar_colors, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Average Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set_ylim(0, 1)

# Annotate the best K
axes[1].annotate(f'Best K={best_k_sil}',
                 xy=(best_k_sil, max(sil_scores)),
                 xytext=(best_k_sil + 1.5, max(sil_scores) - 0.05),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=11, color='red')

plt.suptitle('Choosing K: Elbow Method vs Silhouette Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: Elbow Method

The **elbow plot** (left panel) shows inertia on the y-axis and K on the x-axis.

- **Inertia always decreases as K increases.** At K=1, all points are in one cluster, so inertia is very high. At K=N (one cluster per point), inertia is zero.
- **You cannot just pick the K with the lowest inertia** — that would always give you K=N, which is useless.
- Instead, look for the **"elbow"** — the point where the curve bends sharply and the rate of decrease slows down dramatically. Adding more clusters beyond the elbow doesn't give you much improvement.
- The red dashed line marks K=4, which looks like a reasonable elbow in this dataset.

**Limitation:** The elbow is often subjective. If the curve is smooth with no clear bend, the elbow method alone isn't enough. Always use it alongside the silhouette score.

---

### How to Read This Chart: Silhouette Score

The **silhouette bar chart** (right panel) shows the average silhouette score for each K.

- **Range is -1 to +1.** For K-Means on well-separated data, you typically see values between 0.3 and 0.8.
  - **Score near +1:** Points are tightly packed in their own cluster and far from neighboring clusters. Excellent separation.
  - **Score near 0:** Points are near the boundary between clusters. Ambiguous assignments.
  - **Score near -1:** Points are closer to a different cluster than their own. Wrong cluster.
- **The tallest bar is the best K.** The red bar highlights the optimal K.
- Unlike the elbow method, there is always a clear, unambiguous winner.

> **Silhouette is more reliable than the elbow.** When in doubt, trust silhouette score.

---

## Section 7 — K-Means Instability Demonstration

### Why Does This Matter in Practice?

If you run K-Means with `n_init=1` (a single random initialization), the result depends heavily on where the centroids happened to be placed at the start. Two runs on the exact same data can produce very different cluster assignments.

This is not just a theoretical concern — in a production pipeline, if you retrain your clustering model without fixing the seed, you might see customers silently moving between segments, dashboards showing inconsistent results, or downstream models receiving different inputs.

The fix is simple: **use `n_init >= 10` and always set `random_state`.**

In [ ]:
# Run KMeans(n_clusters=3, n_init=1) 10 times with different seeds
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
inertia_list = []

for i, seed in enumerate(seeds):
    km = KMeans(n_clusters=3, n_init=1, random_state=seed)
    labels = km.fit_predict(X_blobs)
    inertia_list.append(km.inertia_)

    axes[i].scatter(X_blobs[:, 0], X_blobs[:, 1],
                    c=labels, cmap='Set1', s=15, alpha=0.7, edgecolors='none')
    axes[i].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                    marker='X', s=120, c='black', zorder=5)
    axes[i].set_title(f'Seed={seed}\nInertia={km.inertia_:.1f}', fontsize=10)
    axes[i].set_xticks([])
    axes[i].set_yticks([])

plt.suptitle('K-Means Instability: n_init=1, 10 Different Seeds\n'
             'Same data — different results every time',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Inertia range: {min(inertia_list):.2f} — {max(inertia_list):.2f}")
print(f"Spread: {max(inertia_list) - min(inertia_list):.2f}")
print()
print("Fix: Use n_init=10 (or higher) + random_state to always get the best, reproducible result.")

### How to Read This Chart: K-Means Instability

The grid shows **10 identical runs** of K-Means on the same dataset — the only difference is the random seed used for initialization.

- **Each panel** shows the clustering result for one seed. The three colors represent the three predicted clusters.
- **Black X markers** are the final centroid positions.
- **The title** of each panel shows the seed and the resulting inertia.

**What to notice:**
- Some panels show three well-separated, correct clusters. Others show bizarre, incorrect groupings where one true cluster is split and another is merged.
- Panels with higher inertia are finding worse solutions — local minima.
- Even panels with similar inertia can have different cluster colorings due to **label permutation** (the algorithm might call the same physical cluster "cluster 0" in one run and "cluster 2" in another).

> **The takeaway:** `n_init=1` is dangerous in practice. With `n_init=10`, scikit-learn runs the algorithm 10 times with different initializations and returns the best result, giving you the stable, correct clusters shown in the lowest-inertia panels here.

---

## Section 8 — Mini-Batch K-Means

### Why Do We Need It?

Standard K-Means processes **every data point in every iteration**. For small datasets (< 100,000 points), this is fine. But for truly large datasets — millions of transactions, billions of user events, or terabytes of log data — processing every point every time is impractically slow.

**Mini-Batch K-Means** solves this by using a random **mini-batch** of data (typically 1,000–10,000 points) for each update step instead of the full dataset. This means:
- Each iteration is much faster
- Convergence requires fewer passes over the data
- The result is nearly as good as full K-Means for most practical purposes

The trade-off: mini-batch K-Means typically has slightly **higher inertia** (slightly worse clusters) than full K-Means because each update step uses incomplete information. For most applications, this quality loss is negligible compared to the massive speed gain.

In [ ]:
# Generate a large dataset (100,000 points)
X_large, _ = make_blobs(n_samples=100_000, centers=6, cluster_std=1.5, random_state=RANDOM_STATE)
X_large = StandardScaler().fit_transform(X_large)

print(f"Large dataset shape: {X_large.shape}")
print()

# Time full K-Means
t0 = time.time()
km_full = KMeans(n_clusters=6, n_init=3, random_state=RANDOM_STATE)
km_full.fit(X_large)
t_full = time.time() - t0

# Time Mini-Batch K-Means
t0 = time.time()
km_mini = MiniBatchKMeans(n_clusters=6, batch_size=1024, n_init=3, random_state=RANDOM_STATE)
km_mini.fit(X_large)
t_mini = time.time() - t0

speedup = t_full / t_mini if t_mini > 0 else float('inf')

print(f"Full K-Means:")
print(f"  Time   : {t_full:.3f} seconds")
print(f"  Inertia: {km_full.inertia_:.2f}")
print()
print(f"Mini-Batch K-Means (batch_size=1024):")
print(f"  Time   : {t_mini:.3f} seconds")
print(f"  Inertia: {km_mini.inertia_:.2f}")
print()
print(f"Speedup factor: {speedup:.1f}x faster")
inertia_diff_pct = (km_mini.inertia_ - km_full.inertia_) / km_full.inertia_ * 100
print(f"Inertia increase: {inertia_diff_pct:.2f}% (quality penalty for the speed gain)")

In [ ]:
# Plot both clustering results on a 10,000-point sample for visualization
sample_idx = np.random.choice(len(X_large), size=5000, replace=False)
X_sample = X_large[sample_idx]

labels_full_sample = km_full.predict(X_sample)
labels_mini_sample = km_mini.predict(X_sample)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_sample[:, 0], X_sample[:, 1],
                c=labels_full_sample, cmap='tab10', s=5, alpha=0.5, edgecolors='none')
axes[0].scatter(km_full.cluster_centers_[:, 0], km_full.cluster_centers_[:, 1],
                marker='X', s=200, c='black', zorder=5)
axes[0].set_title(f'Full K-Means\nTime: {t_full:.3f}s | Inertia: {km_full.inertia_:.0f}', fontsize=11)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].scatter(X_sample[:, 0], X_sample[:, 1],
                c=labels_mini_sample, cmap='tab10', s=5, alpha=0.5, edgecolors='none')
axes[1].scatter(km_mini.cluster_centers_[:, 0], km_mini.cluster_centers_[:, 1],
                marker='X', s=200, c='black', zorder=5)
axes[1].set_title(f'Mini-Batch K-Means (batch=1024)\nTime: {t_mini:.3f}s | Inertia: {km_mini.inertia_:.0f}', fontsize=11)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.suptitle('Full K-Means vs Mini-Batch K-Means on 100,000 Points',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Output: Speed vs Quality Tradeoff

**The printed output** shows the key numbers side by side:
- **Time:** How long each algorithm took to train on 100,000 points. Mini-Batch is substantially faster.
- **Inertia:** The within-cluster sum of squares. Lower is better. Mini-Batch inertia is slightly higher (worse), but the difference is usually very small (a few percent).
- **Speedup factor:** How many times faster Mini-Batch was.

**The scatter plots** show a 5,000-point sample colored by cluster assignment.
- If both plots look nearly identical — same cluster shapes, similar centroid positions — Mini-Batch is doing its job well.
- Minor differences in cluster boundaries near the edges of clusters are expected and acceptable.
- The black X markers show centroid positions. They should be in nearly the same locations for both algorithms.

> **When to use Mini-Batch:** When your dataset has more than ~50,000 points and training time matters. For datasets under 50k, just use standard K-Means — the speed difference isn't worth the quality penalty.

---

## Section 9 — Cluster Profiling (Business Output)

### From Algorithm Output to Business Insight

Running K-Means gives you cluster labels (0, 1, 2, 3...). But numbers aren't useful to a marketing team or a product manager. **Cluster profiling** is the step where you translate raw cluster assignments into human-understandable descriptions.

The process is simple:
1. Compute the **mean of each feature** for each cluster.
2. Compare the means across clusters to understand what makes each group distinctive.
3. Give each cluster a **descriptive name** based on the profile.

This is the deliverable that goes into presentations, marketing campaigns, and product decisions. The algorithm is just the means; the profile is the end.

In [ ]:
# Compute cluster profiles (mean of each feature per cluster)
features = ['age', 'annual_income', 'spending_score']
profile = df_mall.groupby('cluster')[features].mean().round(1)
profile['size'] = df_mall.groupby('cluster').size()

print("Cluster Profile (mean feature values per cluster):")
print()
print(profile.to_string())
print()

# Display as styled DataFrame
profile.style.background_gradient(cmap='YlOrRd', subset=features)

In [ ]:
# Plot cluster profile heatmap
# Normalize each column to 0-1 for fair color comparison
profile_norm = (profile[features] - profile[features].min()) / \
               (profile[features].max() - profile[features].min())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of normalized profiles
sns.heatmap(profile_norm, annot=profile[features], fmt='.1f',
            cmap='YlOrRd', linewidths=0.5, ax=axes[0],
            cbar_kws={'label': 'Normalized Value (0–1)'})
axes[0].set_title('Cluster Profile Heatmap\n(annotations = actual mean values)', fontsize=11)
axes[0].set_xlabel('Feature')
axes[0].set_ylabel('Cluster')
axes[0].set_yticklabels([f'Cluster {i}' for i in range(4)], rotation=0)

# Bar chart comparison
x = np.arange(len(features))
width = 0.2
palette = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']

for i in range(4):
    axes[1].bar(x + i * width, profile_norm.iloc[i],
                width, label=f'Cluster {i}', color=palette[i], alpha=0.85)

axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(features, fontsize=11)
axes[1].set_ylabel('Normalized Value (0–1)')
axes[1].set_title('Cluster Feature Profiles (Normalized)\nfor Stakeholder Comparison', fontsize=11)
axes[1].legend()

plt.suptitle('Cluster Profiling — Mall Customers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: Cluster Profile Heatmap

**This is what you present to stakeholders.** The heatmap distills the entire clustering result into one clean, interpretable table.

- **Each row** is one cluster. **Each column** is one feature.
- **Cell color** shows the relative value — darker orange/red means higher value relative to other clusters.
- **Cell annotation** shows the actual mean value (e.g., mean age = 45 for that cluster).
- **Normalization** maps each feature to a 0–1 range so that features with different scales (age 18–75 vs income $15k–$140k) can be compared on the same heatmap.

**How to name your clusters:**  
Look at which features are high or low for each cluster and construct a business narrative:
- A cluster with **low income + high spending** = "Young Spenders" — they splurge despite modest income.
- A cluster with **high income + high spending** = "Premium Buyers" — your highest-value customers.
- A cluster with **high income + low spending** = "Careful Earners" — they have money but don't spend it here.
- A cluster with **low income + low spending** = "Budget Shoppers" — price-sensitive, transaction-focused.

> **Pro tip:** Always validate your cluster names with domain experts before presenting. The math tells you *what* is different; domain knowledge tells you *why*.

---

## Section 10 — DBSCAN

### The Core Idea: Density, Not Distance to a Center

K-Means defines clusters by proximity to a centroid — it assumes clusters are roughly **spherical (convex)**. This breaks down when clusters have irregular shapes.

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) takes a completely different approach: a cluster is any **dense region** of data, where "dense" means many points are packed closely together. The shape of the cluster can be anything — crescents, rings, spirals, blobs — as long as the points are dense.

### Two Key Parameters

**`epsilon` (eps):** The maximum distance between two points for them to be considered neighbors. Think of it as the radius of a circle drawn around each point. If another point falls inside that circle, they are neighbors.

**`min_samples`:** The minimum number of points within `epsilon` distance for a point to be considered a **core point** (i.e., a point that can start or extend a cluster). Points with fewer than `min_samples` neighbors are either **border points** (on the edge of a cluster) or **noise**.

### Three Types of Points in DBSCAN

- **Core point:** Has ≥ `min_samples` neighbors within `epsilon`. Can expand a cluster.
- **Border point:** Within `epsilon` of a core point, but has fewer than `min_samples` neighbors itself. Part of a cluster but can't expand it.
- **Noise (outlier):** Not within `epsilon` of any core point. Assigned label **-1**.

> **The -1 label is powerful.** DBSCAN is the only major clustering algorithm that explicitly labels outliers. This makes it ideal for anomaly detection.

### Key Advantages over K-Means
- No need to specify the number of clusters in advance.
- Finds clusters of **arbitrary shape**.
- Explicitly identifies **outliers** as noise.
- Robust to outliers (they don't distort cluster shapes like they do in K-Means).

In [ ]:
# Show K-Means FAILING on moons vs DBSCAN succeeding
km_moons = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE)
labels_km_moons = km_moons.fit_predict(X_moons)

db_moons = DBSCAN(eps=0.3, min_samples=5)
labels_db_moons = db_moons.fit_predict(X_moons)

n_clusters_db = len(set(labels_db_moons)) - (1 if -1 in labels_db_moons else 0)
n_noise_db    = list(labels_db_moons).count(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# K-Means on moons (fails)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1],
                c=labels_km_moons, cmap='Set1', s=25, alpha=0.8, edgecolors='none')
axes[0].scatter(km_moons.cluster_centers_[:, 0], km_moons.cluster_centers_[:, 1],
                marker='X', s=200, c='black', zorder=5)
axes[0].set_title('K-Means on make_moons — FAILS\n(assumes spherical clusters)', fontsize=11)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# DBSCAN on moons (succeeds)
# Color noise points differently
colors_db = np.where(labels_db_moons == -1, 'black',
             np.where(labels_db_moons == 0, '#e41a1c', '#377eb8'))
axes[1].scatter(X_moons[:, 0], X_moons[:, 1],
                c=colors_db, s=25, alpha=0.8, edgecolors='none')
axes[1].set_title(f'DBSCAN on make_moons — SUCCEEDS\n'
                  f'(eps=0.3, min_samples=5 | {n_clusters_db} clusters, {n_noise_db} noise)',
                  fontsize=11)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.suptitle('K-Means vs DBSCAN on Non-Convex Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"DBSCAN found {n_clusters_db} clusters and {n_noise_db} noise points.")

### How to Read This Chart: DBSCAN on Non-Convex Clusters

**Left panel (K-Means fails):** K-Means tries to divide the moons into two spherical clusters by placing centroids. Because the two crescent shapes overlap in the middle, K-Means ends up splitting each crescent in half and merging halves from different crescents — completely wrong.

**Right panel (DBSCAN succeeds):** DBSCAN correctly traces along the crescent shapes because it follows density, not distance to a center. Each crescent is one continuous dense region, so DBSCAN correctly finds two clusters.

- **Red and blue dots** are the two clusters found by DBSCAN.
- **Black dots** are noise points (label = -1) — points too isolated to belong to any cluster.

> **Key insight:** K-Means works by finding the nearest centroid. DBSCAN works by asking "is this region dense?" These two definitions lead to completely different behavior on non-convex data.

In [ ]:
# DBSCAN parameter sensitivity grid: eps=[0.1, 0.3, 0.5] x min_samples=[3, 5, 10]
eps_values = [0.1, 0.3, 0.5]
min_samples_values = [3, 5, 10]

fig, axes = plt.subplots(3, 3, figsize=(13, 12))

for row_idx, min_s in enumerate(min_samples_values):
    for col_idx, eps in enumerate(eps_values):
        ax = axes[row_idx][col_idx]

        db = DBSCAN(eps=eps, min_samples=min_s)
        labels = db.fit_predict(X_moons)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise    = list(labels).count(-1)

        # Create color array: noise = gray, cluster 0 = red, cluster 1 = blue, etc.
        unique_labels = sorted(set(labels))
        cmap = plt.cm.get_cmap('tab10', max(len(unique_labels), 1))
        color_map = {lbl: ('gray' if lbl == -1 else cmap(i % 10))
                     for i, lbl in enumerate(unique_labels)}
        colors = [color_map[l] for l in labels]

        ax.scatter(X_moons[:, 0], X_moons[:, 1],
                   c=colors, s=12, alpha=0.8, edgecolors='none')
        ax.set_title(f'eps={eps}, min_samples={min_s}\n'
                     f'{n_clusters} clusters, {n_noise} noise', fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

        # Label axes for border panels only
        if row_idx == 2:
            ax.set_xlabel(f'eps={eps}', fontsize=10)
        if col_idx == 0:
            ax.set_ylabel(f'min_samples={min_s}', fontsize=10)

plt.suptitle('DBSCAN Parameter Sensitivity Grid on make_moons\n'
             'Gray = noise (label -1)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: DBSCAN Parameter Sensitivity Grid

This 3×3 grid shows all nine combinations of `eps` and `min_samples` on the same dataset.

- **Columns** vary `eps` (the neighborhood radius): 0.1 (tiny), 0.3 (medium), 0.5 (large).
- **Rows** vary `min_samples` (density threshold): 3 (low), 5 (medium), 10 (high).
- **Colors** indicate cluster membership. **Gray** dots are noise (label = -1).

**What happens as you change each parameter:**

**Increasing eps (left → right):**
- Small eps → many points are isolated → lots of noise, possibly too many tiny clusters.
- Large eps → neighborhoods overlap more → clusters merge, eventually everything becomes one cluster.
- There is a "sweet spot" (typically the middle column) where the two moons are correctly identified.

**Increasing min_samples (top → bottom):**
- Low min_samples → almost any point can be a core point → fewer noise points, possibly merges distinct regions.
- High min_samples → only very dense regions form clusters → more noise points, but tighter cluster cores.

**How to choose parameters in practice:**
1. Use the **k-distance graph** (sort points by distance to their k-th nearest neighbor and look for a "knee"). The knee gives a good `eps`.
2. Set `min_samples` to at least the number of dimensions + 1 (a common heuristic).
3. Always sanity-check with a visualization like this grid.

---

## Section 11 — Agglomerative Hierarchical Clustering

### Bottom-Up Merging

**Agglomerative (bottom-up) hierarchical clustering** takes yet another approach:

1. Start with every single data point as its own cluster (N clusters).
2. Find the two closest clusters.
3. Merge them into one cluster.
4. Repeat steps 2–3 until only one cluster remains.

The result is a **tree of merges** called a **dendrogram**. By cutting the tree at a certain height, you get different numbers of clusters.

```
Step 0 (start):  [A] [B] [C] [D] [E]  ← every point is its own cluster
Step 1:          [A,B] [C] [D] [E]    ← A and B are closest; merge
Step 2:          [A,B] [C,D] [E]      ← C and D are closest; merge
Step 3:          [A,B,E] [C,D]        ← E is close to A,B; merge
Step 4:          [A,B,C,D,E]          ← final merge into one cluster
```

### Linkage Criteria

The key question is: **how do you measure the distance between two clusters** (not just two points)? This is called the **linkage criterion**:

| Linkage | Definition | Effect |
|---------|-----------|--------|
| **Ward** | Minimize increase in total within-cluster variance | Creates compact, roughly equal-sized clusters. Usually the best default. |
| **Complete** | Distance = max distance between any two points in the clusters | Creates compact clusters; sensitive to outliers |
| **Average** | Distance = average of all pairwise distances between clusters | Balance between single and complete |
| **Single** | Distance = min distance between any two points (nearest neighbor) | Creates elongated, chain-like clusters; very sensitive to noise |

In [ ]:
# Generate a small sample for the dendrogram (30 points from make_blobs)
X_dendro, y_dendro = make_blobs(n_samples=30, centers=3, cluster_std=0.5, random_state=RANDOM_STATE)
X_dendro = StandardScaler().fit_transform(X_dendro)

# Compute linkage matrix using Ward linkage
Z = linkage(X_dendro, method='ward')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(14, 6))

dendrogram(Z,
           ax=ax,
           leaf_rotation=90,
           leaf_font_size=9,
           color_threshold=3.5)

# Add horizontal cut line at height 3.5 (gives 3 clusters)
ax.axhline(y=3.5, color='red', linestyle='--', linewidth=2, label='Cut here → 3 clusters')
ax.set_xlabel('Data Point Index', fontsize=12)
ax.set_ylabel('Merge Distance (Ward linkage)', fontsize=12)
ax.set_title('Dendrogram — Hierarchical Clustering (Ward Linkage, 30 Points)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("The horizontal red line shows where you 'cut' the tree.")
print("Count the number of vertical lines it intersects — that's the number of clusters.")

### How to Read This Chart: Dendrogram

A **dendrogram** is a tree diagram showing the order and distance at which clusters were merged.

- **The x-axis** shows individual data points (leaves of the tree).
- **The y-axis** shows the **merge distance** — how far apart the two clusters were when they were merged. Higher = they were less similar.
- **Each horizontal line** marks a merge event: the two branches below it were merged into one cluster.
- **Colors** group the branches that will end up in the same cluster when the tree is cut at the red line.

**How to choose K:**
- Draw a horizontal line across the dendrogram (the red dashed line).
- Count how many vertical lines the horizontal line crosses — that's the number of clusters at that cut height.
- **Look for large vertical gaps** — areas where the y-axis distance between two consecutive merges is large. This means merging those clusters would be "costly" (they are genuinely different), so you should cut there.
- In this example, cutting at ~3.5 gives 3 clusters, which matches the data we generated.

> **Key advantage:** The dendrogram gives you the full picture of cluster structure at every possible K simultaneously. You can make an informed decision about K just by looking at the tree.

In [ ]:
# Compare 4 linkage methods on make_blobs (2x2 grid)
linkages = ['ward', 'complete', 'average', 'single']
linkage_titles = {
    'ward':     'Ward — minimize variance increase\n(usually best)',
    'complete': 'Complete — max pairwise distance\n(compact clusters)',
    'average':  'Average — mean pairwise distance\n(balanced)',
    'single':   'Single — min pairwise distance\n(chaining effect)'
}

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for i, link in enumerate(linkages):
    agg = AgglomerativeClustering(n_clusters=3, linkage=link)
    labels = agg.fit_predict(X_blobs)

    axes[i].scatter(X_blobs[:, 0], X_blobs[:, 1],
                    c=labels, cmap='Set1', s=25, alpha=0.8, edgecolors='none')
    axes[i].set_title(linkage_titles[link], fontsize=11)
    axes[i].set_xlabel('Feature 1')
    axes[i].set_ylabel('Feature 2')

plt.suptitle('Agglomerative Clustering: Linkage Method Comparison (k=3 on make_blobs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: Linkage Comparison

The 2×2 grid shows the same make_blobs dataset clustered with four different linkage methods. Since make_blobs creates well-separated spherical clusters, all four methods should give similar results here — but they will diverge significantly on more complex shapes.

- **Ward** creates the most compact, equal-sized clusters. It minimizes the total within-cluster variance at each merge step, which tends to produce the most "balanced" clusters.
- **Complete** also creates compact clusters, but by using the maximum distance between cluster members, it tends to be more sensitive to outliers.
- **Average** provides a middle ground — it averages all pairwise distances, which makes it more robust to outliers than complete, but less strict than Ward.
- **Single** creates elongated "chain" clusters. Because it only looks at the single closest pair of points, one outlier can connect two very different clusters (the "chaining effect"). This is why single linkage is rarely used in practice.

> **Recommendation:** Start with Ward. It works well for most datasets. Switch to Average or Complete if Ward produces unbalanced clusters. Avoid Single unless you specifically need to find elongated, chain-like structures.

---

## Section 12 — Gaussian Mixture Models (GMM)

### Probabilistic Clustering

K-Means makes **hard assignments** — every point belongs to exactly one cluster. But in reality, a point near the boundary between two clusters might plausibly belong to either one. **Gaussian Mixture Models (GMM)** handle this naturally by making **soft assignments** — every point gets a probability of belonging to each cluster.

### How It Works

GMM models the data as a **mixture of K Gaussian (normal) distributions**. Each component in the mixture has:
- A **mean** (center)
- A **covariance matrix** (controls the shape and orientation — not just width, but tilt)
- A **weight** (how much of the data this component explains)

The algorithm (Expectation-Maximization, or EM) alternates between:
1. **E-step:** Estimate the probability that each point belongs to each component.
2. **M-step:** Update the component parameters (mean, covariance, weight) to best explain those probabilities.

### Why GMM Beats K-Means on Elliptical Clusters

K-Means assumes all clusters are **spherical** (equal covariances). GMM can model **elliptical clusters** — rotated, stretched, or skewed in any direction. This is a critical advantage when real-world clusters don't happen to be round.

### Choosing the Number of Components: BIC

GMM uses the **Bayesian Information Criterion (BIC)** instead of the elbow/silhouette. BIC balances model fit (how well the Gaussian mixture explains the data) against model complexity (how many parameters are used). **Lower BIC = better model.** The optimal number of components is where BIC reaches its minimum.

In [ ]:
# Generate elongated/rotated elliptical clusters to challenge K-Means
np.random.seed(RANDOM_STATE)

# Create three clusters with different orientations and elongations
n = 200
# Cluster 0: elongated horizontally
C0 = np.random.randn(n, 2) @ np.array([[2.5, 0.5], [0.2, 0.4]]) + np.array([0, 4])
# Cluster 1: elongated diagonally
C1 = np.random.randn(n, 2) @ np.array([[0.4, -1.5], [1.5, 0.4]]) + np.array([4, 0])
# Cluster 2: compact but tilted
C2 = np.random.randn(n, 2) @ np.array([[0.6, 0.8], [-0.8, 0.6]]) + np.array([-4, -2])

X_elliptical = np.vstack([C0, C1, C2])
y_elliptical_true = np.array([0]*n + [1]*n + [2]*n)

X_elliptical = StandardScaler().fit_transform(X_elliptical)

print(f"Elliptical dataset shape: {X_elliptical.shape}")

# Fit K-Means and GMM
km_ell  = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
gmm_ell = GaussianMixture(n_components=3, covariance_type='full', random_state=RANDOM_STATE)

labels_km_ell  = km_ell.fit_predict(X_elliptical)
labels_gmm_ell = gmm_ell.fit_predict(X_elliptical)

# Silhouette scores
sil_km  = silhouette_score(X_elliptical, labels_km_ell)
sil_gmm = silhouette_score(X_elliptical, labels_gmm_ell)

print(f"K-Means  silhouette: {sil_km:.4f}")
print(f"GMM      silhouette: {sil_gmm:.4f}")

In [ ]:
# Plot K-Means vs GMM on elliptical clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means
axes[0].scatter(X_elliptical[:, 0], X_elliptical[:, 1],
                c=labels_km_ell, cmap='Set1', s=20, alpha=0.7, edgecolors='none')
axes[0].scatter(km_ell.cluster_centers_[:, 0], km_ell.cluster_centers_[:, 1],
                marker='X', s=200, c='black', zorder=5)
axes[0].set_title(f'K-Means — Struggles with ellipses\nSilhouette: {sil_km:.4f}', fontsize=11)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# GMM with confidence ellipses
axes[1].scatter(X_elliptical[:, 0], X_elliptical[:, 1],
                c=labels_gmm_ell, cmap='Set1', s=20, alpha=0.7, edgecolors='none')

# Draw confidence ellipses for each GMM component
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

def draw_confidence_ellipse(ax, mean, cov, n_std=2.0, color='black', alpha=0.3):
    """Draw a covariance confidence ellipse."""
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(eigenvalues)
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle,
                      edgecolor=color, facecolor=color, alpha=alpha, linewidth=2)
    ax.add_patch(ellipse)

palette_ell = ['#e41a1c', '#377eb8', '#4daf4a']
for k in range(3):
    draw_confidence_ellipse(axes[1],
                            mean=gmm_ell.means_[k],
                            cov=gmm_ell.covariances_[k],
                            n_std=2.0,
                            color=palette_ell[k],
                            alpha=0.2)

axes[1].set_title(f'GMM — Handles ellipses correctly\nSilhouette: {sil_gmm:.4f}', fontsize=11)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.suptitle('K-Means vs GMM on Elliptical Clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: GMM Confidence Ellipses

**Left panel (K-Means):** K-Means draws spherical boundaries between clusters. When clusters are elongated, K-Means often cuts through the middle of a cluster along the wrong axis, forcing points from the same cluster into different groups.

**Right panel (GMM):** Each colored ellipse shows the **2-standard-deviation confidence ellipse** for one GMM component. This represents the region that contains about 95% of the probability mass for that Gaussian component.

- **Shape of the ellipse** reflects the covariance of the cluster — stretched ellipses mean the cluster is elongated.
- **Angle of the ellipse** shows the orientation — a tilted ellipse means the cluster is rotated in feature space.
- When ellipses match the actual spread of points, GMM has correctly learned the cluster geometry.

> **When to use GMM over K-Means:** When your clusters are not spherical — elongated, tilted, or variable in density. GMM is particularly useful for signal processing, finance (returns distributions), and any domain where data naturally follows a Gaussian distribution.

In [ ]:
# BIC score for choosing number of GMM components (2 to 10)
n_components_range = range(1, 11)
bic_scores  = []
aic_scores  = []

for n in n_components_range:
    gmm = GaussianMixture(n_components=n, covariance_type='full',
                          random_state=RANDOM_STATE, max_iter=200)
    gmm.fit(X_elliptical)
    bic_scores.append(gmm.bic(X_elliptical))
    aic_scores.append(gmm.aic(X_elliptical))

best_n_bic = list(n_components_range)[bic_scores.index(min(bic_scores))]
best_n_aic = list(n_components_range)[aic_scores.index(min(aic_scores))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(n_components_range), bic_scores, 'o-', color='steelblue',
        linewidth=2, markersize=8, label='BIC')
ax.plot(list(n_components_range), aic_scores, 's--', color='coral',
        linewidth=2, markersize=8, label='AIC')
ax.axvline(x=best_n_bic, color='steelblue', linestyle=':', alpha=0.7,
           label=f'Best BIC: n={best_n_bic}')
ax.axvline(x=best_n_aic, color='coral', linestyle=':', alpha=0.7,
           label=f'Best AIC: n={best_n_aic}')
ax.set_xlabel('Number of Components (K)', fontsize=12)
ax.set_ylabel('Score (lower is better)', fontsize=12)
ax.set_title('GMM — BIC and AIC vs Number of Components', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(list(n_components_range))

plt.tight_layout()
plt.show()

print(f"Best number of components by BIC: {best_n_bic}")
print(f"Best number of components by AIC: {best_n_aic}")
print()
print("BIC penalizes complexity more than AIC.")
print("Prefer BIC when you want a simpler, more parsimonious model.")

### How to Read This Chart: BIC Score vs Components

The BIC (Bayesian Information Criterion) and AIC (Akaike Information Criterion) are both **penalized likelihood scores** that balance how well the model fits the data against how many parameters it uses.

- **Lower BIC/AIC = better.** You want the model that explains the data well without using unnecessary components.
- **The curve typically drops steeply at first** (adding more components genuinely improves the fit), then **levels off or rises** (more components don't help and the complexity penalty dominates).
- **The minimum** of the curve is the optimal number of components.
- **BIC penalizes complexity more heavily** than AIC (it has a stronger penalty term for the number of parameters). BIC tends to favor simpler models; AIC tends to favor slightly more complex ones.

**Why this matters over the elbow method:**  
The BIC has a proper statistical foundation. It's not just a heuristic visual inspection — there's a clear mathematical minimum. This makes it more objective and easier to explain to stakeholders.

> **Rule of thumb:** Use BIC for model selection when you care about having a parsimonious model. Use AIC when you care more about predictive accuracy.

---

## Section 13 — Algorithm Comparison on All Datasets

### The Grand Comparison

Now we put all four algorithms side by side against all three benchmark datasets. This 4×3 grid is the most important visualization in this notebook — it shows at a glance which algorithms work on which data shapes.

- **Rows:** K-Means, DBSCAN, Agglomerative, GMM
- **Columns:** make_blobs, make_moons, make_circles

No single algorithm wins everywhere. The right choice depends entirely on the shape and structure of your data.

In [ ]:
# Define algorithms and datasets
algorithms = [
    ('K-Means',          KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE)),
    ('DBSCAN',           DBSCAN(eps=0.3, min_samples=5)),
    ('Agglomerative',    AgglomerativeClustering(n_clusters=2, linkage='ward')),
    ('GMM',              GaussianMixture(n_components=2, covariance_type='full',
                                         random_state=RANDOM_STATE)),
]

datasets_compare = [
    ('make_blobs\n(spherical, separated)', X_blobs,   3),
    ('make_moons\n(crescents)',            X_moons,   2),
    ('make_circles\n(concentric rings)',   X_circles, 2),
]

fig, axes = plt.subplots(4, 3, figsize=(14, 17))

for row_idx, (alg_name, alg) in enumerate(algorithms):
    for col_idx, (ds_name, X_ds, k_true) in enumerate(datasets_compare):
        ax = axes[row_idx][col_idx]

        # Handle k=3 for blobs
        if k_true == 3:
            if hasattr(alg, 'n_clusters'):
                alg.set_params(n_clusters=3)
            elif hasattr(alg, 'n_components'):
                alg.set_params(n_components=3)
        else:
            if hasattr(alg, 'n_clusters'):
                alg.set_params(n_clusters=2)
            elif hasattr(alg, 'n_components'):
                alg.set_params(n_components=2)

        try:
            labels = alg.fit_predict(X_ds)
        except Exception:
            labels = np.zeros(len(X_ds), dtype=int)

        # Mark noise for DBSCAN
        n_noise = list(labels).count(-1) if -1 in labels else 0

        unique_labels = sorted(set(labels))
        cmap = plt.cm.get_cmap('Set1', max(len(unique_labels), 2))
        color_map = {lbl: ('lightgray' if lbl == -1 else cmap(i % 10))
                     for i, lbl in enumerate(unique_labels)}
        colors = [color_map[l] for l in labels]

        ax.scatter(X_ds[:, 0], X_ds[:, 1],
                   c=colors, s=12, alpha=0.8, edgecolors='none')

        # Add centroids for K-Means and GMM
        if hasattr(alg, 'cluster_centers_'):
            ax.scatter(alg.cluster_centers_[:, 0], alg.cluster_centers_[:, 1],
                       marker='X', s=100, c='black', zorder=5)
        elif hasattr(alg, 'means_'):
            ax.scatter(alg.means_[:, 0], alg.means_[:, 1],
                       marker='X', s=100, c='black', zorder=5)

        noise_text = f" ({n_noise} noise)" if n_noise > 0 else ""
        ax.set_title(f'{alg_name} on {ds_name}{noise_text}', fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

        # Mark failing cases with a red border
        fails = (alg_name == 'K-Means'     and col_idx > 0) or \
                (alg_name == 'Agglomerative' and col_idx > 0 and 'ward' in str(alg))

plt.suptitle('Algorithm Comparison: 4 Algorithms × 3 Datasets\n'
             'Gray = noise | X = centroid/mean',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: Algorithm Comparison Grid

Each cell in the 4×3 grid shows how one algorithm performs on one dataset.

- **Colors** indicate cluster assignments. **Gray dots** (DBSCAN only) are noise points (label = -1).
- **Black X markers** show centroids (K-Means) or component means (GMM).

**Reading the rows:**
- **K-Means (row 1):** Works perfectly on blobs. Completely fails on moons and circles because it assumes spherical clusters and uses centroid distance.
- **DBSCAN (row 2):** Struggles a bit on well-separated blobs (might create noise), but shines on moons and circles because it follows density, not shape.
- **Agglomerative (row 3):** Ward linkage works well on blobs. Performance on moons and circles depends on linkage — single linkage would succeed but Ward tends to fail on non-convex shapes.
- **GMM (row 4):** Excellent on blobs and elliptical patterns. Struggles with moons and circles because it assumes Gaussian distributions.

**Reading the columns:**
- **make_blobs (col 1):** Every algorithm succeeds here. This is not a useful discriminator.
- **make_moons (col 2):** Only DBSCAN reliably finds the two crescents. Others cut through the wrong axis.
- **make_circles (col 3):** Only DBSCAN handles concentric rings. All other algorithms divide the space along a straight line.

> **The big takeaway:** There is no universally best algorithm. Before clustering, always look at your data and ask: "Are my clusters likely to be spherical, elongated, crescent-shaped, or some other shape?" Let the answer guide your algorithm choice.

---

## Section 14 — Evaluation Metrics Summary

### Three Metrics, Three Perspectives

Because clustering is unsupervised, we can't simply ask "how many did you get right?" Instead, we use **internal evaluation metrics** that measure the geometric quality of the clusters without needing ground-truth labels:

| Metric | Measures | Range | Best Value |
|--------|----------|-------|------------|
| **Silhouette Score** | How similar each point is to its own cluster vs other clusters | -1 to +1 | Higher is better |
| **Davies-Bouldin Index** | Average ratio of within-cluster scatter to between-cluster separation | 0 to ∞ | Lower is better |
| **Calinski-Harabasz Score** | Ratio of between-cluster dispersion to within-cluster dispersion | 0 to ∞ | Higher is better |

All three metrics agree on what "good" clusters look like: **tight within clusters, far apart between clusters.** But they emphasize different aspects and can sometimes disagree, especially when cluster shapes are non-convex.

In [ ]:
# Compute all three metrics for 4 algorithms on make_blobs
# Re-run each algorithm with k=3 on make_blobs

eval_algorithms = {
    'K-Means':        KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE),
    'DBSCAN':         DBSCAN(eps=0.5, min_samples=5),
    'Agglomerative':  AgglomerativeClustering(n_clusters=3, linkage='ward'),
    'GMM':            GaussianMixture(n_components=3, covariance_type='full',
                                      random_state=RANDOM_STATE),
}

eval_results = []

for name, alg in eval_algorithms.items():
    labels = alg.fit_predict(X_blobs)

    # Skip metrics if DBSCAN labels all as noise or produces only 1 cluster
    unique_labels = set(labels) - {-1}
    if len(unique_labels) < 2:
        eval_results.append({
            'Algorithm':         name,
            'Silhouette (↑)':    np.nan,
            'Davies-Bouldin (↓)': np.nan,
            'Calinski-Harabasz (↑)': np.nan,
            'N Clusters':        len(unique_labels),
            'N Noise':           list(labels).count(-1)
        })
        continue

    # Exclude noise points from metric calculation
    mask = labels != -1
    X_eval = X_blobs[mask]
    l_eval  = labels[mask]

    sil  = silhouette_score(X_eval, l_eval)
    db   = davies_bouldin_score(X_eval, l_eval)
    ch   = calinski_harabasz_score(X_eval, l_eval)

    eval_results.append({
        'Algorithm':              name,
        'Silhouette (↑)':         round(sil, 4),
        'Davies-Bouldin (↓)':     round(db, 4),
        'Calinski-Harabasz (↑)':  round(ch, 2),
        'N Clusters':             len(unique_labels),
        'N Noise':                list(labels).count(-1)
    })

df_eval = pd.DataFrame(eval_results).set_index('Algorithm')
print("Clustering Evaluation Metrics on make_blobs (n=300, 3 centers):")
print()
print(df_eval.to_string())
print()
print("↑ = higher is better    ↓ = lower is better")

In [ ]:
# Styled visualization of the metrics table
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

algorithms_list = df_eval.index.tolist()
metrics = [
    ('Silhouette (↑)',          True,  'steelblue', 'Higher = better cluster separation'),
    ('Davies-Bouldin (↓)',      False, 'coral',     'Lower = tighter, more separated clusters'),
    ('Calinski-Harabasz (↑)',   True,  'seagreen',  'Higher = denser, more separated clusters'),
]

for ax, (metric, higher_better, color, subtitle) in zip(axes, metrics):
    vals = df_eval[metric].fillna(0).values
    bars = ax.bar(algorithms_list, vals, color=color, alpha=0.8, edgecolor='white')

    # Highlight best bar
    best_idx = int(np.argmax(vals) if higher_better else np.argmin(vals))
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2.5)

    ax.set_title(f'{metric}\n{subtitle}', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.set_xticklabels(algorithms_list, rotation=15, ha='right', fontsize=10)

    # Add value labels on bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01 * max(vals),
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Clustering Evaluation Metrics Comparison on make_blobs\n'
             'Black border = best algorithm for that metric',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Table: Clustering Evaluation Metrics

The table and bar charts show the same metrics in two formats. Use the table for precise numbers; use the charts for a quick visual comparison.

**Silhouette Score (↑ higher is better):**
- Ranges from -1 to +1.
- Measures whether each point is closer to points in its own cluster than to points in the nearest other cluster.
- Values above 0.5 generally indicate well-separated clusters. Values near 0 mean clusters are overlapping. Negative values mean points are closer to another cluster.
- Weakness: biased toward convex, equally-sized clusters. DBSCAN with irregular shapes can score deceptively low.

**Davies-Bouldin Index (↓ lower is better):**
- Measures the average "worst-case" similarity between each cluster and its most similar neighbor cluster. Similarity is defined as the ratio of within-cluster scatter to between-cluster distance.
- Perfect score = 0 (no overlap, infinite separation).
- Doesn't require knowing the true number of clusters, so it's useful for the same K-selection task as silhouette.

**Calinski-Harabasz Score (↑ higher is better):**
- Also called the Variance Ratio Criterion. Ratio of between-cluster dispersion (how spread out the centroids are) to within-cluster dispersion (how spread out points are within clusters).
- No upper bound — the exact value only matters for comparison between configurations.
- Very fast to compute and tends to peak sharply at the correct K on well-separated data.

> **Which metric to trust?** Use multiple metrics together. If they all agree, you can be confident. If they disagree, look at the visualization — the metric may not match the problem's geometry.

---

## Section 15 — Summary and Key Takeaways

### Algorithm Selection Guide

Use this table as your cheat sheet when deciding which clustering algorithm to reach for first:

| Algorithm | When to Use | Strengths | Limitations |
|-----------|------------|-----------|-------------|
| **K-Means** | Large datasets with roughly spherical, similar-sized clusters. Customer segmentation, image compression. | Fast, scalable, simple to understand and explain. Centroids are intuitive business outputs. | Requires K upfront. Fails on non-convex shapes, unequal-density clusters. Sensitive to outliers. Must scale features. |
| **Mini-Batch K-Means** | Same as K-Means but dataset > 100,000 rows. Streaming data. | 5–20× faster than K-Means with minimal quality loss. | Slightly higher inertia than full K-Means. Same shape limitations. |
| **DBSCAN** | Irregular shapes, unknown K, presence of noise/outliers. Anomaly detection. Geospatial data. | Finds arbitrary shapes. No K needed. Explicitly labels outliers. Robust to noise. | Very sensitive to eps and min_samples. Struggles with varying density. Slow on very large datasets. |
| **Agglomerative** | When you want to explore multiple values of K at once. Small-to-medium datasets. Hierarchical data. | Dendrogram reveals full structure. No K needed upfront. Multiple linkage options. | Slow (O(n² log n) with Ward). Hard to scale to > 10,000 points. Memory intensive. |
| **GMM** | Elliptical clusters. When soft/probabilistic assignments are needed. Mixture modeling. | Models elliptical shapes. Returns probabilities. BIC for model selection. Most flexible distribution assumption. | Requires K upfront. Slow to converge. Can collapse on degenerate covariances (needs regularization). |

---

### Decision Flowchart

```
START
  │
  ├─ Do you know K (number of clusters)?
  │   ├─ YES → Do clusters have irregular shapes?
  │   │         ├─ NO  → Are clusters elongated/elliptical?
  │   │         │         ├─ YES → GMM
  │   │         │         └─ NO  → Is dataset very large (>100k)?
  │   │         │                   ├─ YES → Mini-Batch K-Means
  │   │         │                   └─ NO  → K-Means
  │   │         └─ YES → DBSCAN or Agglomerative (single/average linkage)
  │   └─ NO → Do you want a hierarchy / explore multiple K at once?
  │            ├─ YES → Agglomerative Hierarchical (use dendrogram to choose K)
  │            └─ NO  → Are there outliers/noise in the data?
  │                       ├─ YES → DBSCAN
  │                       └─ NO  → Use Elbow + Silhouette → K-Means or GMM
  │
END
```

---

### Common Mistakes to Avoid

**1. Not scaling features before K-Means**  
K-Means uses Euclidean distance. If one feature is measured in thousands (income: \$20k–\$120k) and another in ones (age: 18–75), the income will completely dominate the distance calculation. Always use `StandardScaler` before K-Means (and most other clustering algorithms).

**2. Using Euclidean distance in high dimensions**  
In very high-dimensional spaces (hundreds or thousands of features), all points become roughly equidistant — this is called the **curse of dimensionality**. Euclidean distance becomes meaningless. Apply dimensionality reduction (PCA, UMAP) before clustering high-dimensional data.

**3. Ignoring the -1 label in DBSCAN**  
DBSCAN assigns -1 to noise points. If you blindly pass these labels to downstream models or evaluation metrics without handling them, you'll get incorrect results. Always check `(labels == -1).sum()` after DBSCAN.

**4. Choosing K based on inertia alone**  
Inertia always decreases as K increases. The elbow method is subjective and sometimes has no visible elbow. Always confirm with silhouette score (or BIC for GMM).

**5. Running K-Means with n_init=1**  
As demonstrated in Section 7, single-initialization K-Means can converge to poor local minima. Always use `n_init >= 10` in production.

**6. Forgetting to set random_state**  
Without a fixed random state, clustering results will be different on every run. This makes your pipelines non-reproducible and your results hard to debug or report. Always set `random_state` on any stochastic algorithm.

---

### Evaluation Metrics Cheat Sheet

| Metric | Formula Intuition | Range | Best | Use When |
|--------|------------------|-------|------|----------|
| **Silhouette** | (b-a) / max(a,b) where a=within-cluster dist, b=nearest-cluster dist | -1 to +1 | +1 | General purpose, convex clusters |
| **Davies-Bouldin** | Mean of max(σᵢ + σⱼ) / d(cᵢ,cⱼ) | 0 to ∞ | 0 | When clusters are roughly spherical |
| **Calinski-Harabasz** | Between-cluster / within-cluster dispersion ratio | 0 to ∞ | ∞ | When you want a fast, sharp K indicator |
| **BIC/AIC** | Penalized log-likelihood | -∞ to +∞ | Min | GMM model selection only |
| **Inertia** | Sum of squared distances to nearest centroid | 0 to ∞ | 0 | Elbow method for K-Means K selection |